# Benchmark de descarga Rigol MSO2102A

Este notebook compara la descarga legacy de `oscrigol.getchannels` con la descarga optimizada `getchannelsFast`. El foco no es solo medir tiempo: cada iteracion valida que se descarguen `MDEPTH` muestras y que la forma de onda tenga amplitud suficiente.

Setup de señal de prueba usado para estas mediciones:

- Generador: DSS Function Generator 60 MHz MCP MPF3060.
- Modo: señal arbitraria.
- Salida: Canal B-Arb.
- Tipo de señal: B31 Earthquake.
- BURST activado.
- Carrier Frequency: 20 kHz.
- CHB Amplitude: 1 Vpp.
- Burst N Cycles: 1 Cycle.
- Duty: 50 %.
- Conexion: salida del generador al CH1 del Rigol MSO2102A.
- Ajuste esperado: la señal se muestra en la totalidad de la pantalla del osciloscopio.


## 1. Seguridad antes de tocar el equipo

Por defecto, este notebook no abre comunicacion con el osciloscopio. Cambiar `RUN_HARDWARE` a `True` solo cuando el Rigol y el generador esten listos.

Si el raw socket no responde, probar primero con `TRANSPORT = "instr"`. El modo socket requiere que el Rigol acepte conexiones en el puerto 5555.

In [ ]:
RUN_HARDWARE = True  # Cambiar a True para medir con el Rigol real

IP_ADDRESS = "192.168.2.2"
TRANSPORT = "socket"  # "instr" o "socket"
VISA_BACKEND = "nivisa"  # "pyvisa" usa pyvisa-py; "nivisa" usa la biblioteca NI-VISA del sistema

CHANNELS = (1,)
MDEPTH = 70000
ITERATIONS = 32

TRIGGER_SOURCE = "CHAN1"
TRIGGER_LEVEL = 0.1  # 100 mV
TRIGGER_COUPLING = "AC"
TRIGGER_SLOPE = "POS"

VERTICAL_SCALE = 0.05  # 50 mV/div
HORIZONTAL_SCALE = 10e-6  # 10 us/div

POLL_INTERVAL = 0.005
TRIGGER_TIMEOUT = 5.0
WAVEFORM_DELAY = 0.01
WAVEFORM_POINTS_TIMEOUT = 0.5
ARM_DELAY = 0.05
MAX_RETRIES_FAST = 2
MIN_VPP_FAST = None  # rechazo dentro de getchannelsFast; ejemplo: 0.05
SIGNAL_VPP_THRESHOLD = 0.05  # reporte de calidad del benchmark; ajustar segun la senal real

METHODS_TO_BENCHMARK = [("fast", "getchannelsFast")]


## 2. Imports y carga de la clase

Como la carpeta `rigol-MSO2102A` tiene un guion en el nombre, cargamos `oscrigol.py` por path usando `importlib`.

In [ ]:
import csv
import importlib.util
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
if HERE.name != "rigol-MSO2102A":
    HERE = Path("oscilloscopes/rigol-MSO2102A").resolve()

OSCRIGOL_PATH = HERE / "oscrigol.py"
spec = importlib.util.spec_from_file_location("oscrigol_module", OSCRIGOL_PATH)
oscrigol_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(oscrigol_module)
oscrigol = oscrigol_module.oscrigol

OSCRIGOL_PATH

## 3. Funciones auxiliares

`build_scope` crea el objeto con `INSTR` o `SOCKET`. `prepare_scope` configura trigger, canal, escalas, adquisicion normal y profundidad de memoria.

Nota: para comparar de forma justa, se prepara el osciloscopio antes de cada bloque de pruebas.

In [ ]:
def build_scope(transport="instr", download_mode="legacy", visa_backend=VISA_BACKEND):
    scope = oscrigol(
        ip_address=IP_ADDRESS,
        use_socket=(transport == "socket"),
        socket_port=5555,
        visa_backend=visa_backend,
    )
    scope.config(
        channels=CHANNELS,
        chanBand=tuple(["20M"] * len(CHANNELS)),
        chanCoup=tuple(["AC"] * len(CHANNELS)),
        chanInv=tuple(["OFF"] * len(CHANNELS)),
        chanImp=tuple(["OMEG"] * len(CHANNELS)),
        trigSource=TRIGGER_SOURCE,
        trigCoup=TRIGGER_COUPLING,
        trigLevel=TRIGGER_LEVEL,
        trigSlope=TRIGGER_SLOPE,
        acquisition=1,
        mdepth=MDEPTH,
        download_mode=download_mode,
    )
    return scope


def prepare_scope(scope):
    scope.initComm()
    print(scope.getID().strip())
    scope.setEdgeTrigger(TRIGGER_SOURCE, TRIGGER_SLOPE, TRIGGER_COUPLING, TRIGGER_LEVEL)
    for channel in CHANNELS:
        scope.setChannel(channel, "20M", "AC", "OFF", "OMEG")
        scope.setVertScale(channel, VERTICAL_SCALE)
    scope.setHorScale(HORIZONTAL_SCALE)
    scope.run()
    scope.setSampAcquisition()
    check = scope.setandcheckmdepth(MDEPTH)
    if check:
        scope.closeComm()
        raise RuntimeError("El Rigol rechazo la profundidad de memoria solicitada")
    return scope


def close_scope(scope):
    try:
        scope.closeComm()
    except Exception as exc:
        print(f"No se pudo cerrar la comunicacion limpiamente: {exc}")


def waveform_metrics(values, expected_samples=MDEPTH, vpp_threshold=SIGNAL_VPP_THRESHOLD):
    y = np.asarray(values).reshape(-1)
    samples = int(y.size)
    samples_ok = samples == int(expected_samples)
    if samples == 0:
        vpp = 0.0
        rms = 0.0
        std = 0.0
    else:
        vpp = float(np.ptp(y))
        rms = float(np.sqrt(np.mean(y**2)))
        std = float(np.std(y))
    signal_ok = samples_ok and (vpp_threshold is None or vpp >= vpp_threshold)
    return {
        "samples": samples,
        "samples_ok": samples_ok,
        "vpp_v": vpp,
        "rms_v": rms,
        "std_v": std,
        "signal_ok": signal_ok,
    }


## 4. Prueba rapida de conexion

Esta celda solo abre comunicacion, consulta `*IDN?` y cierra. Conviene correrla antes del benchmark.

In [ ]:
if RUN_HARDWARE:
    scope = build_scope(TRANSPORT)
    try:
        scope.initComm()
        print(scope.getID())
    finally:
        close_scope(scope)
else:
    print("RUN_HARDWARE = False; no se abre comunicacion con el Rigol.")

## 5. Una adquisicion legacy y una fast


In [ ]:
single_results = {}

single_methods = [("legacy", "getchannels"), ("fast", "getchannelsFast")]

if RUN_HARDWARE:
    for mode, method_name in single_methods:
        scope = build_scope(TRANSPORT, download_mode=mode, visa_backend=VISA_BACKEND)
        try:
            prepare_scope(scope)
            method = getattr(scope, method_name)
            t0 = time.perf_counter()
            if method_name == "getchannelsFast":
                values, vmax = method(
                    CHANNELS,
                    MDEPTH,
                    poll_interval=POLL_INTERVAL,
                    trigger_timeout=TRIGGER_TIMEOUT,
                    waveform_delay=WAVEFORM_DELAY,
                    arm_delay=ARM_DELAY,
                    max_retries=MAX_RETRIES_FAST,
                    min_vpp=MIN_VPP_FAST,
                    waveform_points_timeout=WAVEFORM_POINTS_TIMEOUT,
                )
            else:
                values, vmax = method(CHANNELS, MDEPTH)
            elapsed = time.perf_counter() - t0
            values = np.asarray(values)
            metrics = waveform_metrics(values)
            time_s = scope.getHorValues(metrics["samples"])
            attempts = int(getattr(scope, "_last_acquisition_attempts", 1))
            single_results[method_name] = {
                "transport": TRANSPORT,
                "visa_backend": VISA_BACKEND,
                "method": method_name,
                "values": values,
                "time_s": np.asarray(time_s),
                "vmax": float(vmax),
                "elapsed_s": elapsed,
                "attempts": attempts,
                "retries": max(0, attempts - 1),
                **metrics,
            }
            print(
                f"{method_name}: {elapsed:.3f} s, samples={metrics['samples']}, "
                f"Vpp={metrics['vpp_v']*1e3:.1f} mV, retries={max(0, attempts - 1)}, "
                f"vmax_ch2={vmax:.4g}"
            )
        except Exception as exc:
            print(f"{method_name}: fallo la adquisicion de prueba: {exc}")
        finally:
            close_scope(scope)
else:
    print("RUN_HARDWARE = False; saltando adquisicion real.")


## 6. Visualizacion de la adquisicion de prueba

Grafica las senales capturadas en la celda anterior usando el eje temporal consultado al osciloscopio con `getHorValues`, que a su vez lee `:TIMebase:SCALe?`, `:TIMebase:OFFSet?` y `:ACQuire:SRATe?`.


In [ ]:
if single_results:
    plt.figure(figsize=(10, 4))
    for method_name, row in single_results.items():
        y = row["values"].reshape(-1)
        t = row["time_s"].reshape(-1)
        if y.size == 0:
            print(f"{method_name}: lectura vacia, no se grafica")
            continue
        if t.size != y.size:
            print(f"{method_name}: eje temporal {t.size} no coincide con senal {y.size}; no se grafica")
            continue
        elapsed = row["elapsed_s"]
        plt.plot(t * 1e6, y * 1e3, label=f"{method_name} ({elapsed:.3f} s, n={y.size})")
    plt.xlabel("Tiempo [us]")
    plt.ylabel("CH1 [mV]")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.show()
else:
    print("No hay resultados para graficar todavia.")


## 7. Benchmark legacy vs fast

Corre `ITERATIONS` adquisiciones con cada metodo seleccionado. Cada fila mide una llamada completa y reporta tiempo, muestras, Vpp, intentos y reintentos.


In [ ]:
def call_method(scope, method_name, method):
    if method_name == "getchannelsFast":
        return method(
            CHANNELS,
            MDEPTH,
            poll_interval=POLL_INTERVAL,
            trigger_timeout=TRIGGER_TIMEOUT,
            waveform_delay=WAVEFORM_DELAY,
            arm_delay=ARM_DELAY,
            max_retries=MAX_RETRIES_FAST,
            min_vpp=MIN_VPP_FAST,
            waveform_points_timeout=WAVEFORM_POINTS_TIMEOUT,
        )
    return method(CHANNELS, MDEPTH)


def run_benchmark(
    transport="instr",
    visa_backend=VISA_BACKEND,
    iterations=10,
    keep_waveforms=True,
    methods=METHODS_TO_BENCHMARK,
):
    rows = []
    waveforms = []
    for mode, method_name in methods:
        scope = build_scope(transport, download_mode=mode, visa_backend=visa_backend)
        try:
            prepare_scope(scope)
            method = getattr(scope, method_name)
            for idx in range(iterations):
                t0 = time.perf_counter()
                error = ""
                values = np.array([])
                vmax = np.nan
                try:
                    values, vmax = call_method(scope, method_name, method)
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}"
                elapsed = time.perf_counter() - t0
                values = np.asarray(values)
                metrics = waveform_metrics(values)
                attempts = int(getattr(scope, "_last_acquisition_attempts", 1))
                retries = max(0, attempts - 1)
                if not error and not metrics["samples_ok"]:
                    error = f"samples={metrics['samples']} esperado={MDEPTH}"
                if not error and not metrics["signal_ok"]:
                    error = f"Vpp={metrics['vpp_v']:.6g} V debajo del umbral"

                row = {
                    "transport": transport,
                    "visa_backend": visa_backend,
                    "method": method_name,
                    "iteration": idx + 1,
                    "elapsed_s": elapsed,
                    "attempts": attempts,
                    "retries": retries,
                    "vmax_ch2": float(vmax),
                    "error": error,
                    **metrics,
                }
                rows.append(row)

                if keep_waveforms and not error:
                    time_s = scope.getHorValues(metrics["samples"])
                    waveforms.append({
                        **row,
                        "values": values.reshape(-1).copy(),
                        "time_s": np.asarray(time_s).reshape(-1).copy(),
                    })

                status = "OK" if not error else "ERROR"
                print(
                    f"{transport}/{visa_backend} {method_name} {idx + 1}/{iterations}: "
                    f"{elapsed:.3f} s, samples={metrics['samples']}, "
                    f"Vpp={metrics['vpp_v']*1e3:.1f} mV, retries={retries}, {status}"
                )
        finally:
            close_scope(scope)
    return rows, waveforms


if RUN_HARDWARE:
    benchmark_rows, benchmark_waveforms = run_benchmark(TRANSPORT, VISA_BACKEND, ITERATIONS)
else:
    benchmark_rows = []
    benchmark_waveforms = []
    print("RUN_HARDWARE = False; saltando benchmark real.")

benchmark_rows[:3]


## 8. Superposicion de mediciones por metodo

Esta celda superpone todas las formas de onda medidas por cada metodo. Sirve para verificar visualmente que las variantes rapidas no esten devolviendo otro evento, ruido, una ventana temporal distinta o una lectura corrupta.

La linea negra es el promedio de las iteraciones de cada metodo.

In [ ]:
def group_label(row):
    return f"{row['transport']} / {row['visa_backend']} / {row['method']}"


def plot_waveform_overlay(waveforms):
    groups = sorted({group_label(row) for row in waveforms})
    if not groups:
        print("No hay formas de onda validas para graficar.")
        return

    fig, axes = plt.subplots(len(groups), 1, figsize=(11, 3.2 * len(groups)), sharex=True, sharey=True)
    if len(groups) == 1:
        axes = [axes]

    for ax, label in zip(axes, groups):
        group_waveforms = [row for row in waveforms if group_label(row) == label]
        traces = []
        times = []
        for row in group_waveforms:
            y = row["values"]
            t = row["time_s"]
            if y.size == 0 or t.size != y.size:
                continue
            alpha = 0.22 if row["retries"] == 0 else 0.5
            linewidth = 0.8 if row["retries"] == 0 else 1.2
            ax.plot(t * 1e6, y * 1e3, alpha=alpha, linewidth=linewidth)
            traces.append(y)
            times.append(t)

        if traces:
            min_len = min(trace.size for trace in traces)
            stack = np.vstack([trace[:min_len] for trace in traces])
            t_mean = times[0][:min_len]
            ax.plot(t_mean * 1e6, stack.mean(axis=0) * 1e3, color="black", linewidth=2.0, label="promedio")
            ax.legend(loc="best")

        retried = sum(row["retries"] > 0 for row in group_waveforms)
        ax.set_title(f"{label} - {len(group_waveforms)} validas, {retried} con retry")
        ax.set_ylabel("CH1 [mV]")
        ax.grid(True, linestyle="--", alpha=0.35)

    axes[-1].set_xlabel("Tiempo [us]")
    fig.tight_layout()
    plt.show()


if benchmark_waveforms:
    plot_waveform_overlay(benchmark_waveforms)
else:
    print("No hay formas de onda guardadas. Ejecutar primero la celda del benchmark con keep_waveforms=True.")


## 9. Conteo de senales validas vs ruido

Esta celda clasifica cada captura usando un umbral de tension pico a pico. La idea es detectar rapidamente cuantas iteraciones parecen contener la senal del generador y cuantas parecen ser solo ruido/base.

Ajustar `SIGNAL_VPP_THRESHOLD` segun la amplitud esperada. Por ejemplo, `0.05` equivale a 50 mVpp.

In [ ]:
def print_signal_noise_summary(rows):
    groups = {}
    for row in rows:
        key = (row["transport"], row["visa_backend"], row["method"])
        groups.setdefault(key, []).append(row)

    for (transport, visa_backend, method), group_rows in groups.items():
        n_total = len(group_rows)
        n_ok = sum(row["signal_ok"] and row["samples_ok"] and not row["error"] for row in group_rows)
        n_bad_samples = sum(not row["samples_ok"] for row in group_rows)
        n_bad_signal = sum(row["samples_ok"] and not row["signal_ok"] for row in group_rows)
        n_errors = sum(bool(row["error"]) for row in group_rows)
        retries = sum(row["retries"] for row in group_rows)
        vpps = [row["vpp_v"] for row in group_rows if row["samples"] > 0]
        vpp_text = "sin Vpp" if not vpps else (
            f"Vpp mediana={statistics.median(vpps)*1e3:.1f} mV, "
            f"min={min(vpps)*1e3:.1f} mV, max={max(vpps)*1e3:.1f} mV"
        )
        print(
            f"{transport}/{visa_backend} {method}: "
            f"validas={n_ok}/{n_total}, errores={n_errors}, "
            f"samples_mal={n_bad_samples}, senal_baja={n_bad_signal}, "
            f"retries_total={retries}, {vpp_text}"
        )


if benchmark_rows:
    print_signal_noise_summary(benchmark_rows)
else:
    print("No hay datos. Ejecutar primero la celda del benchmark.")


## 10. Evolucion temporal del benchmark

Arriba se grafica el tiempo de cada iteracion. Abajo se grafica el tiempo acumulado por metodo; el ultimo punto es el tiempo total empleado por esas iteraciones.

In [ ]:
def plot_timing_evolution(rows):
    groups = sorted({(row["transport"], row["visa_backend"], row["method"]) for row in rows})
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 7), sharex=True)

    for transport, visa_backend, method in groups:
        group_rows = [
            row for row in rows
            if (row["transport"], row["visa_backend"], row["method"]) == (transport, visa_backend, method)
        ]
        group_rows = sorted(group_rows, key=lambda row: row["iteration"])
        if not group_rows:
            continue
        iterations = np.array([row["iteration"] for row in group_rows])
        elapsed = np.array([row["elapsed_s"] for row in group_rows])
        cumulative = np.cumsum(elapsed)
        label = f"{transport}/{visa_backend}/{method}"

        axes[0].plot(iterations, elapsed, marker="o", label=label)
        axes[1].plot(iterations, cumulative, marker="o", label=label)
        axes[1].annotate(
            f"total {cumulative[-1]:.1f} s",
            xy=(iterations[-1], cumulative[-1]),
            xytext=(6, 0),
            textcoords="offset points",
            va="center",
        )

    axes[0].set_ylabel("Tiempo por iteracion [s]")
    axes[0].grid(True, linestyle="--", alpha=0.35)
    axes[0].legend(loc="best")

    axes[1].set_xlabel("Iteracion")
    axes[1].set_ylabel("Tiempo acumulado [s]")
    axes[1].grid(True, linestyle="--", alpha=0.35)
    axes[1].legend(loc="best")

    fig.tight_layout()
    plt.show()


def plot_retries(rows):
    groups = sorted({(row["transport"], row["visa_backend"], row["method"]) for row in rows})
    plt.figure(figsize=(10.5, 3.5))
    for transport, visa_backend, method in groups:
        group_rows = sorted([
            row for row in rows
            if (row["transport"], row["visa_backend"], row["method"]) == (transport, visa_backend, method)
        ], key=lambda row: row["iteration"])
        plt.step(
            [row["iteration"] for row in group_rows],
            [row["retries"] for row in group_rows],
            where="mid",
            marker="o",
            label=f"{transport}/{visa_backend}/{method}",
        )
    plt.xlabel("Iteracion")
    plt.ylabel("Reintentos")
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(loc="best")
    plt.show()


if benchmark_rows:
    plot_timing_evolution(benchmark_rows)
    plot_retries(benchmark_rows)
else:
    print("No hay datos temporales. Ejecutar primero la celda del benchmark.")


## 11. Resumen y guardado

La mediana suele ser mas representativa que el promedio si hay algun trigger o paquete de red atipico.

In [ ]:
def summarize_rows(rows):
    groups = {}
    for row in rows:
        key = (row["transport"], row["visa_backend"], row["method"])
        groups.setdefault(key, []).append(row)

    summary = []
    for (transport, visa_backend, method), group_rows in groups.items():
        valid_rows = [row for row in group_rows if not row["error"] and row["samples_ok"] and row["signal_ok"]]
        elapsed = [row["elapsed_s"] for row in valid_rows]
        summary.append({
            "transport": transport,
            "visa_backend": visa_backend,
            "method": method,
            "count": len(group_rows),
            "valid": len(valid_rows),
            "failures": len(group_rows) - len(valid_rows),
            "retried_iterations": sum(row["retries"] > 0 for row in group_rows),
            "retries_total": sum(row["retries"] for row in group_rows),
            "mean_s": statistics.mean(elapsed) if elapsed else np.nan,
            "median_s": statistics.median(elapsed) if elapsed else np.nan,
            "min_s": min(elapsed) if elapsed else np.nan,
            "max_s": max(elapsed) if elapsed else np.nan,
            "std_s": statistics.stdev(elapsed) if len(elapsed) > 1 else 0.0,
        })
    return summary


if benchmark_rows:
    summary = summarize_rows(benchmark_rows)
    for row in summary:
        print(
            f"{row['transport']}/{row['visa_backend']} {row['method']}: "
            f"validas={row['valid']}/{row['count']} failures={row['failures']} "
            f"median={row['median_s']:.3f}s mean={row['mean_s']:.3f}s "
            f"min={row['min_s']:.3f}s max={row['max_s']:.3f}s "
            f"iter_con_retry={row['retried_iterations']} retries_total={row['retries_total']}"
        )

    backend_label = str(VISA_BACKEND).replace("/", "_")
    output_path = HERE / f"benchmark_oscrigol_{TRANSPORT}_{backend_label}_ch1.csv"
    with open(output_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=benchmark_rows[0].keys())
        writer.writeheader()
        writer.writerows(benchmark_rows)
    print(f"Resultados guardados en: {output_path}")
else:
    print("No hay datos de benchmark para resumir.")


## 12. Comparar INSTR vs SOCKET

Cuando la prueba con `TRANSPORT = "instr"` funcione, repetir con `TRANSPORT = "socket"`. Tambien se puede correr la celda siguiente para medir ambos transportes en una sola tanda.

Usar esta celda solo si el Rigol responde bien por ambos caminos.

In [ ]:
RUN_COMPARISON = True
COMPARISON_CASES = [
    {"transport": "socket", "visa_backend": "nivisa", "label": "socket + NI-VISA"},
    {"transport": "socket", "visa_backend": "pyvisa", "label": "socket + pyvisa-py"},
]
COMPARISON_METHODS = [("fast", "getchannelsFast")]

if RUN_HARDWARE and RUN_COMPARISON:
    comparison_rows = []
    comparison_waveforms = []
    for case in COMPARISON_CASES:
        print(f"\n=== {case['label']} ===")
        rows, waves = run_benchmark(
            transport=case["transport"],
            visa_backend=case["visa_backend"],
            iterations=ITERATIONS,
            keep_waveforms=True,
            methods=COMPARISON_METHODS,
        )
        for row in rows:
            row["case"] = case["label"]
        for row in waves:
            row["case"] = case["label"]
        comparison_rows.extend(rows)
        comparison_waveforms.extend(waves)

    print("\nResumen comparativo")
    for row in summarize_rows(comparison_rows):
        print(
            f"{row['transport']}/{row['visa_backend']} {row['method']}: "
            f"validas={row['valid']}/{row['count']} failures={row['failures']} "
            f"median={row['median_s']:.3f}s mean={row['mean_s']:.3f}s "
            f"max={row['max_s']:.3f}s retries_total={row['retries_total']}"
        )

    output_path = HERE / "benchmark_oscrigol_socket_backend_comparison_ch1.csv"
    with open(output_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=comparison_rows[0].keys())
        writer.writeheader()
        writer.writerows(comparison_rows)
    print(f"Comparacion guardada en: {output_path}")

    plot_timing_evolution(comparison_rows)
    plot_retries(comparison_rows)
    plot_waveform_overlay(comparison_waveforms)
    print_signal_noise_summary(comparison_rows)
else:
    print("Celda desactivada. Cambiar RUN_COMPARISON a True para comparar backends/transporte.")


## Interpretacion esperada

- `socket/nivisa/getchannelsFast` suele dar la menor mediana de tiempo cuando la biblioteca NI-VISA esta instalada y estable.
- `socket/pyvisa/getchannelsFast` suele ser algo mas lento, pero puede ser mas predecible en algunas corridas.
- Una iteracion solo se considera valida si `samples == MDEPTH`, no hay error y `signal_ok` es verdadero.
- Los reintentos son aceptables si son pocos: preservan la calidad de datos, pero aumentan el tiempo total.
- Si aparecen errores o muestras incompletas, revisar primero `WAVEFORM_POINTS_TIMEOUT`, `MAX_RETRIES_FAST`, el nivel de trigger y que la señal del generador ocupe la pantalla completa.
